In [ ]:
pip install langchain

In [ ]:
# Install the langchain-community package if it's not already installed
!pip install -U langchain-community

# Import TextLoader from the correct location
from langchain_community.document_loaders import TextLoader

In [ ]:
loader = TextLoader("nvda_news_1.txt")
data = loader.load()
print(data[0].metadata)

{'source': 'nvda_news_1.txt'}


In [ ]:
pip install -qU langchain-community beautifulsoup4

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader([
        "https://www.moneycontrol.com/news/business/banks/hdfc-bank-re-appoints-sanmoy-chakrabarti-as-chief-risk-officer-11259771.html",
        "https://www.moneycontrol.com/news/business/markets/market-corrects-post-rbi-ups-inflation-forecast-icrr-bet-on-these-top-10-rate-sensitive-stocks-ideas-11142611.html",
        "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
    ])
data = loader.load()
print(len(data))
#print(data[1])

3


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(data)
print(len(docs))


72


In [ ]:
print(docs[33])

page_content='| Stop-Loss: Rs 131 | Target: Rs 155 | Return: 9 percentSince February 2022, Manappuram Finance has been in a consolidation phase. After reaching a low point at Rs 81 levels in June 2022, the stock gradually climbed. During its ascent, it surpassed the 50-day, 100-day, and 200-day moving averages. However, its upward momentum slowed around Rs 133-134 levels, acting as a resistance, and a corrective decline followed. This decline found support near the 200-day moving average. A rebound in the price followed this price action.In the past month, the price tested the resistance at Rs 131-133 levels multiple times. Finally, a breakout occurred which took the price above the horizontal trendline resistance of Rs 133, creating fresh buying opportunities.Presently, one can hold the stock, maintain a stop-loss near Rs 131, while expecting an upward move towards Rs 155.L&T Finance Holdings: Buy | LTP: Rs 126.2 | Stop-Loss: Rs 119 | Target: Rs 138-140 | Return: 11 percentAfter formi

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
# Taking some random text from wikipedia

text = """Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is embroiled in a catastrophic blight and famine, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for humankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007 and was originally set to be directed by Steven Spielberg.
Kip Thorne, a Caltech theoretical physicist and 2017 Nobel laureate in Physics,[4] was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm. Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects, and the company Double Negative created additional digital effects.

Interstellar premiered in Los Angeles on October 26, 2014. In the United States, it was first released on film stock, expanding to venues using digital projectors. The film received generally positive reviews from critics and grossed over $677 million worldwide ($715 million after subsequent re-releases), making it the tenth-highest-grossing film of 2014.
It has been praised by astronomers for its scientific accuracy and portrayal of theoretical astrophysics.[5][6][7] Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades."""

In [ ]:
import getpass
import os

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

Enter your Google API key: ··········


In [ ]:
pip install -qU langchain-google-genai

In [ ]:
pip install faiss-cpu

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vectorindex_google_ai = FAISS.from_documents(docs, embeddings)

In [ ]:
import pickle

# Correct way to save FAISS index
# This will create 'vector_index.faiss' and 'vector_index.pkl' in the current directory
vectorindex_google_ai.save_local(folder_path=".", index_name="vector_index")

In [ ]:
loaded_vectorindex = FAISS.load_local(folder_path=".", embeddings=embeddings, index_name="vector_index", allow_dangerous_deserialization=True)

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Fetch API key securely from user data
api_key = ""

# Configure the generative AI library with the API key
genai.configure(api_key=api_key)

# Initialize the Google Generative AI LLM with the gemini-pro model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key)

print("LLM initialized successfully with gemini-pro model!")

LLM initialized successfully with gemini-pro model!


In [ ]:
!pip install -U langchain-classic


In [ ]:
# Change from 'langchain.chains' to 'langchain_classic.chains'
from langchain_classic.chains import RetrievalQAWithSourcesChain


In [ ]:
chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=loaded_vectorindex.as_retriever())
chain

RetrievalQAWithSourcesChain(verbose=False, combine_documents_chain=MapReduceDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Use the following portion of a long document to see if any of the text is relevant to answer the question.\nReturn any relevant text verbatim.\n{context}\nQuestion: {question}\nRelevant text, if any:'), llm=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client 

In [ ]:
query = "what is the price of Tiago iCNG?"

# 4. Run the chain using .invoke() - the modern way
# Note: This chain specifically requires the key "question"
result = chain.invoke({"question": query})

# 5. Print the results clearly
print("ANSWER:\n", result.get("answer"))
print("\nSOURCES:\n", result.get("sources"))

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 54.403113788s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
query = "what is the price of Tiago iCNG?"
# query = "what are the main features of punch iCNG?"

langchain.debug=True

chain({"question": query}, return_only_outputs=True)

{'answer': 'I am sorry, but the provided text does not contain any information about the price of Tiago iCNG.\n',
 'sources': 'https://www.moneycontrol.com/news/business/banks/hdfc-bank-re-appoints-sanmoy-chakrabarti-as-chief-risk-officer-11259771.html'}